# BDD-OIA

This notebook shows an implementation of [BDD-OIA](https://twizwei.github.io/bddoia_project/) task. The BDD-OIA dataset comprises frames extracted from driving scene videos, which are used for autonomous driving predictions. Each frame is annotated with 4 binary labels, indicating the possible actions, namely $\textsf{move\_forward}$, $\textsf{stop}$, $\textsf{turn\_left}$, $\textsf{turn\_right}$. Each frame is also annotated with 21 intermediate binary concepts such as $\textsf{red\_light}$, $\textsf{road\_clear}$, etc., underlying the reasons for the possible actions.

The objective is to predict possible actions for each frame. During training, we only make use of the label supervision, along with a knowledge base, which comprises information about the relations between concepts and actions, e.g., $\textsf{red\_light} \lor \textsf{traffic\_sign} \lor \textsf{obstacle} \implies \textsf{stop}$. The training set consists of 16,000 frames, while the test set contains 4,500 annotated data points.

Before usage, the dataset was pre-processed by [Marconato et al. (2023)](https://proceedings.neurips.cc/paper_files/paper/2023/file/e560202b6e779a82478edb46c6f8f4dd-Paper-Conference.pdf) using a pretrained Faster-RCNN model on BDD-100k, in conjunction with the first module in CBM-AUC [(Sawada & Nakamura, 2022)](https://arxiv.org/abs/2202.01459), resulting in embeddings of dimension 2048.

In [4]:
# Import necessary libraries and modules
import os.path as osp  

import numpy as np
import torch
import torch.nn as nn
from torch import optim

from ablkit.data.evaluation import SymbolAccuracy
from ablkit.reasoning import Reasoner
from ablkit.utils import ABLLogger, print_log

from models.nn import ConceptNet
from models.bdd_nn import BDDNN
from models.bdd_model import BDDABLModel
from reasoning.bddkb import BDDKB
from dataset.data_util import get_dataset
from bridge import BDDBridge
from metric import BDDReasoningMetric

ImportError: cannot import name 'M' from 'ablkit.bridge.base_bridge' (D:\yang\Projects\ABLkit\ablkit\bridge\base_bridge.py)

## Working with Data

First, we get the training and testing datasets:

In [2]:
train_data = get_dataset(fname="train.npz", get_pseudo_label=True)
val_data = get_dataset(fname="val.npz", get_pseudo_label=True)
test_data = get_dataset(fname="test.npz", get_pseudo_label=True)

## Building the Learning Part

To build the learning part, we need to first build a machine learning base model. We use ConceptNet, and encapsulate it within a `BDDNN` object to create the base model. `BDDNN` is a class that encapsulates a PyTorch model, transforming it into a base model with an sklearn-style interface.

In [ ]:
cls = ConceptNet()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(cls.parameters(), lr=0.002)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.002,
    pct_start=0.15,
    epochs=2,
    steps_per_epoch=int(1 / 0.01) + 1,
)

base_model = BDDNN(
    cls,
    loss_fn,
    optimizer,
    scheduler=scheduler,
    device=device,
    batch_size=32,
    num_epochs=1,
)

However, the base model built above deals with instance-level data, and can not directly deal with example-level data. Therefore, we wrap the base model into `BDDABLModel`, a subclass of `ABLModel`, which enables the learning part to train, test, and predict on example-level data.

In [ ]:
model = BDDABLModel(base_model)

## Building the Reasoning Part

In the reasoning part, we first build a knowledge base which contains information about the relations between concepts and actions, e.g., $\text{red\_light} \lor \text{traffic\_sign} \lor \text{obstacle} \implies \text{stop}$. The knowledge base is built in the `BDDKB` class within file `reasoning/bddkb.py`, and is derived from the `KBBase` class.

In [ ]:
kb = BDDKB()

Then, we create a reasoner by instantiating the class `Reasoner`. Due to the indeterminism of abductive reasoning, there could be multiple candidates compatible with the knowledge base. When this happens, reasoner can minimize inconsistencies between the knowledge base and pseudo-labels predicted by the learning part, and then return only one candidate that has the highest consistency. The inconsistency is calculated by the function function `multi_label_confidence_dist`.

In [ ]:
def multi_label_confidence_dist(data_example, candidates, candidates_idxs, reasoning_results):
    pred_prob = data_example.pred_prob.T  # nc x 1
    pred_prob = np.concatenate([1 - pred_prob, pred_prob], axis=1)  # nc x 2
    cols = np.arange(len(candidates_idxs[0]))[None, :]
    corr_prob = pred_prob[cols, candidates_idxs]
    costs = -np.sum(np.log(corr_prob + 1e-6), axis=1)
    return costs

reasoner = Reasoner(
    kb,
    dist_func=multi_label_confidence_dist,
    max_revision=3,
    require_more_revision=3,
)

## Building Evaluation Metrics

Next, we set up evaluation metrics. These metrics will be used to evaluate the model performance during training and testing. Specifically, we use `SymbolAccuracy` and `BDDReasoningMetric`, which are used to evaluate the accuracy of the machine learning model’s predictions and the accuracy of the final reasoning results, respectively.

In [ ]:
metric_list = [SymbolAccuracy(prefix="bdd_oia"), BDDReasoningMetric(kb=kb, prefix="bdd_oia")]

## Bridging Learning and Reasoning

Now, the last step is to bridge the learning and reasoning part. We proceed with this step by creating an instance of `BDDBridge`, which is derived from `SimpleBridge`.

In [2]:
bridge = BDDBridge(model, reasoner, metric_list)

NameError: name 'BDDBridge' is not defined

Perform training and testing by invoking the `train` and `test` methods of `SimpleBridge`.

In [14]:
# Build logger
print_log("Abductive Learning on the MNIST Addition example.", logger="current")
log_dir = ABLLogger.get_current_instance().log_dir
weights_dir = osp.join(log_dir, "weights")

bridge.train(train_data, loops=2, segment_size=0.01, save_interval=1, save_dir=weights_dir)
bridge.test(test_data)